# Workshop 1: Bare states, dressed states, and labels

In a composite quantum system, such as a transmon coupled to a resonator, there are two different ways to describe states.

Bare basis:
- transmon state index
- resonator Fock state index

Example bare label:
(1, 2)
meaning transmon level 1 and resonator photon number 2.

But when the subsystems interact, the true eigenstates of the total Hamiltonian are dressed states.
A dressed state is generally a superposition of many bare product states.

The main question is:

How do we assign a useful bare-style label to each dressed eigenstate?

This notebook only builds intuition. We will not use branch analysis yet.

In [1]:
import scqubits as scq
import numpy as np
import matplotlib.pyplot as plt

## Step 1. Build a simple transmon + resonator system

We keep dimensions small so that the state structure is easy to inspect.

In [2]:
tmon = scq.Transmon(
    EJ=30.0,
    EC=1.2,
    ng=0.0,
    ncut=31,
    truncated_dim=3
)

res = scq.Oscillator(
    E_osc=6.0,
    truncated_dim=5
)

hilbertspace = scq.HilbertSpace([tmon, res])

## Step 2. Add interaction

We use a simple capacitive-style coupling through the transmon charge operator and resonator quadrature.

In [ ]:
g = 0.25

hilbertspace.add_interaction(
    g_strength=g,
    op1=tmon.n_operator,
    op2=res.creation_operator,
    add_hc=True # Adding the Hermitian conjugate of the interaction term. 
)

## Step 3. Diagonalize the full interacting Hamiltonian

In [6]:
evals, evecs = hilbertspace.eigensys(evals_count=10)

print("Lowest 10 dressed energies:")
for idx, energy in enumerate(evals):
    print(f"{idx:2d}  {energy:.6f}")

Lowest 10 dressed energies:
 0  -21.829093
 1  -15.836713
 2  -9.844332
 3  -6.158902
 4  -3.851936
 5  -0.167172
 6  2.152200
 7  5.824609
 8  7.952560
 9  11.816441


## Step 4. Compare with bare product-state indexing

scqubits has a bare indexing convention for the composite Hilbert space.
Let us inspect the mapping from a flat bare index to a tuple label.

In [9]:
hilbertspace.generate_lookup()

print("Bare index table:")
for bare_flat in range(tmon.truncated_dim * res.truncated_dim):
    print(f"{bare_flat:2d} -> {hilbertspace.bare_index(bare_flat)}")

Bare index table:
 0 -> (0, 0)
 1 -> (0, 1)
 2 -> (0, 2)
 3 -> (1, 0)
 4 -> (0, 3)
 5 -> (1, 1)
 6 -> (0, 4)
 7 -> (1, 2)
 8 -> (2, 0)
 9 -> (1, 3)
10 -> (2, 1)
11 -> (1, 4)
12 -> (2, 2)
13 -> (2, 3)
14 -> (2, 4)


## Step 5. Generate a default lookup table

We first use the default dressed-energy style lookup just to inspect the mapping.

In [10]:
hilbertspace.generate_lookup()

print("Dressed index to bare label:")
for dressed_index in range(10):
    print(f"{dressed_index:2d} -> {hilbertspace.dressed_index(dressed_index)}")

Dressed index to bare label:
 0 -> None
 1 -> None
 2 -> None
 3 -> None
 4 -> None
 5 -> None
 6 -> None
 7 -> None
 8 -> None
 9 -> None


In [13]:
for t in range(tmon.truncated_dim):
    for r in range(res.truncated_dim):
        print((t,r), "->", hilbertspace.dressed_index((t,r)))

(0, 0) -> 0
(0, 1) -> 1
(0, 2) -> 2
(0, 3) -> 4
(0, 4) -> 6
(1, 0) -> 3
(1, 1) -> 5
(1, 2) -> 7
(1, 3) -> 9
(1, 4) -> 11
(2, 0) -> 8
(2, 1) -> 10
(2, 2) -> 12
(2, 3) -> 13
(2, 4) -> 14


Depending on the scqubits version, the exact lookup-access method may differ.
If the printed output is confusing, inspect the object with:

- `dir(hilbertspace)`
- `hilbertspace.lookup`
- `help(hilbertspace.generate_lookup)`

The important conceptual point is:
the dressed eigenstates are being assigned bare-style labels somehow.

In [12]:
# Optional: inspect available attributes
[x for x in dir(hilbertspace) if "lookup" in x.lower() or "index" in x.lower()]

['_generate_lookup_by_overlap',
 '_lookup_exists',
 'bare_index',
 'dressed_index',
 'energy_by_bare_index',
 'energy_by_dressed_index',
 'generate_lookup',
 'get_subsys_index',
 'lookup_exists',
 'set_npindextuple']

## Exercises

1. Increase the coupling `g` from 0.25 to 0.8 and diagonalize again.
   What changes in the spectrum?

2. Increase the resonator truncation from 5 to 8.
   Do the lowest few energies change significantly?

3. Write down the difference between:
   - a bare label such as (1, 2)
   - a dressed eigenstate index such as k = 7

## Takeaway

You should now understand:

- bare states are subsystem product states
- dressed states are eigenstates of the interacting Hamiltonian
- the difficult problem is assigning meaningful bare-style labels to dressed states